# Tarea 01 - Deteccion de objetos y velocidad en video

Notebook para detectar un objeto en movimiento comparando cada frame contra el frame de referencia (frame 0 / fondo).

**Pipeline:**
1. Cargar video y extraer todos los frames
2. Calcular diferencia absoluta contra el frame de referencia
3. Umbralizar + operaciones morfologicas para limpiar la mascara
4. Detectar contornos y dibujar bounding box sobre el objeto
5. Precalcular resultados para todos los frames
6. Generar GIF animado con la deteccion

## Importaciones y configuracion


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from pathlib import Path
from IPython.display import display

%matplotlib inline

## Configuracion

Parametros del pipeline en un solo lugar para facilitar el ajuste.

In [ ]:
# --- Ruta al video ---
VIDEO_PATH = Path("video.mp4")
assert VIDEO_PATH.exists(), f"No se encontro el archivo de video en: {VIDEO_PATH.resolve()}"

# --- Parametros de procesamiento ---
NUEVO_ANCHO = 680          # Redimensionar para acelerar procesamiento
NUEVO_ALTO = 480
UMBRAL_DIFF = 40           # Umbral para binarizar la diferencia absoluta
AREA_MINIMA = 500          # Area minima (px^2) para considerar un contorno como objeto
KERNEL_SIZE = (7, 7)       # Tamano del kernel morfologico
ITER_DILATE = 3            # Iteraciones de dilatacion
ITER_ERODE = 1             # Iteraciones de erosion

print(f"Video: {VIDEO_PATH.resolve()}")
print(f"Umbral de diferencia: {UMBRAL_DIFF}")
print(f"Area minima de contorno: {AREA_MINIMA} px^2")

## Carga del video

Se leen todos los frames a memoria (el video es corto, ~5 seg / 162 frames).  
Se redimensionan para acelerar el procesamiento posterior.

In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), "Error: no se pudo abrir el video."

# Propiedades del video
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
ancho_orig = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto_orig = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Frames totales : {total_frames}")
print(f"FPS            : {fps:.2f}")
print(f"Resolucion     : {ancho_orig}x{alto_orig} px")
print(f"Duracion       : {total_frames / fps:.2f} seg")

# Leer todos los frames redimensionados
frames_bgr = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frames_bgr.append(cv2.resize(frame, (NUEVO_ANCHO, NUEVO_ALTO)))
cap.release()

print(f"\nSe cargaron {len(frames_bgr)} frames redimensionados a {NUEVO_ANCHO}x{NUEVO_ALTO} px")

## Precalculo del pipeline de deteccion

Para cada frame se calcula:
1. **Diferencia absoluta** en escala de grises contra el frame 0 (fondo)
2. **Umbralización** binaria con umbral fijo
3. **Morfología** (erosion + dilatacion) para eliminar ruido y cerrar huecos
4. **Contornos** y bounding box del objeto mas grande

Se almacena todo en listas para generar el GIF y las graficas de trayectoria.

In [ ]:
# Frame de referencia (fondo) en escala de grises
frame_ref_gray = cv2.cvtColor(frames_bgr[0], cv2.COLOR_BGR2GRAY)

# Kernel para operaciones morfologicas
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, KERNEL_SIZE)

# Listas para almacenar resultados precalculados
mascaras = []          # Mascara binaria limpia por frame
bboxes = []            # Bounding box (x, y, w, h) del objeto detectado o None
frames_rgb = []        # Frame original en RGB (para matplotlib)

for i, frame_bgr in enumerate(frames_bgr):
    # 0. Convertir a RGB para visualizacion
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    frames_rgb.append(frame_rgb)

    # 1. Diferencia absoluta en escala de grises
    frame_gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    diff = cv2.absdiff(frame_ref_gray, frame_gray)

    # 2. Umbralización binaria
    _, thresh = cv2.threshold(diff, UMBRAL_DIFF, 255, cv2.THRESH_BINARY)

    # 3. Morfología: erosion para quitar ruido, luego dilatacion para cerrar huecos
    limpia = cv2.erode(thresh, kernel, iterations=ITER_ERODE)
    limpia = cv2.dilate(limpia, kernel, iterations=ITER_DILATE)
    mascaras.append(limpia)

    # 4. Encontrar contornos y quedarse con el mas grande (el objeto)
    contornos, _ = cv2.findContours(limpia, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Filtrar por area minima y tomar el contorno mas grande
    contornos_validos = [c for c in contornos if cv2.contourArea(c) >= AREA_MINIMA]
    if contornos_validos:
        mayor = max(contornos_validos, key=cv2.contourArea)
        bboxes.append(cv2.boundingRect(mayor))
    else:
        bboxes.append(None)

# Resumen
detectados = sum(1 for b in bboxes if b is not None)
print(f"Pipeline completado: {len(frames_bgr)} frames procesados")
print(f"Objeto detectado en {detectados}/{len(frames_bgr)} frames")

## Trayectoria del objeto

Grafica de la posicion X del centro del bounding box a lo largo del tiempo.  
Esto es la base para calcular la velocidad del objeto en frames futuros.

In [ ]:
plt.close('all')

# Extraer centros del bounding box donde hubo deteccion
centros_x = []
centros_y = []
frames_con_obj = []

for i, bbox in enumerate(bboxes):
    if bbox is not None:
        x, y, w, h = bbox
        centros_x.append(x + w // 2)
        centros_y.append(y + h // 2)
        frames_con_obj.append(i)

# Graficar trayectoria X e Y vs frame
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(frames_con_obj, centros_x, 'o-', color='tab:blue', markersize=3)
ax1.set_ylabel("Posicion X (px)")
ax1.set_title("Trayectoria del objeto detectado")
ax1.grid(True, alpha=0.3)

ax2.plot(frames_con_obj, centros_y, 'o-', color='tab:orange', markersize=3)
ax2.set_ylabel("Posicion Y (px)")
ax2.set_xlabel("Frame")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Objeto detectado en frames {frames_con_obj[0]} a {frames_con_obj[-1]}" if frames_con_obj else "No se detecto objeto en ningun frame")

## Calibración de escala: metros por píxel a y=286

Se usan dos líneas de referencia horizontales a distintas profundidades de la escena
para calcular el factor de escala (metros/píxel) mediante **interpolación lineal**.

Debido a la perspectiva, objetos más lejanos (y menor en la imagen) abarcan más metros
por píxel que objetos cercanos.

In [ ]:
plt.close('all')
%matplotlib inline

# =============================================================================
# Definir las dos líneas de referencia conocidas
# =============================================================================
ref = [
    {"y": 345, "x0": 351, "x1": 593, "metros": 2.286},
    {"y": 260, "x0": 329, "x1": 543, "metros": 5.8928},
]

# Calcular longitud en píxeles y escala (metros/píxel) para cada referencia
for r in ref:
    r["px"] = r["x1"] - r["x0"]
    r["escala"] = r["metros"] / r["px"]  # metros por píxel

print("=" * 60)
print("ESCALAS DE REFERENCIA")
print("=" * 60)
for r in ref:
    print(f"  y={r['y']:>3d}:  {r['metros']:.4f} m / {r['px']} px = {r['escala']:.6f} m/px")

# =============================================================================
# Interpolación lineal para y=286
# =============================================================================
y_objetivo = 286

# Asignar variables para claridad
y1, escala_1 = ref[1]["y"], ref[1]["escala"]   # y=260 (más lejos, escala mayor)
y2, escala_2 = ref[0]["y"], ref[0]["escala"]    # y=345 (más cerca, escala menor)

# Fórmula de interpolación lineal:
#   escala(y) = escala_260 + (escala_345 - escala_260) * (y - 260) / (345 - 260)
escala_286 = escala_1 + (escala_2 - escala_1) * (y_objetivo - y1) / (y2 - y1)

print(f"\n{'=' * 60}")
print("INTERPOLACIÓN LINEAL")
print(f"{'=' * 60}")
print(f"  Fórmula: escala(y) = escala_260 + (escala_345 - escala_260) × (y - 260) / (345 - 260)")
print(f"  escala({y_objetivo}) = {escala_1:.6f} + ({escala_2:.6f} - {escala_1:.6f}) × ({y_objetivo} - {y1}) / ({y2} - {y1})")
print(f"  escala({y_objetivo}) = {escala_1:.6f} + ({escala_2 - escala_1:.6f}) × ({y_objetivo - y1} / {y2 - y1})")
print(f"  escala({y_objetivo}) = {escala_1:.6f} + ({(escala_2 - escala_1) * (y_objetivo - y1) / (y2 - y1):.6f})")
print(f"\n  >>> Factor de escala en y={y_objetivo}: {escala_286:.6f} m/px <<<")

# =============================================================================
# Visualización sobre el frame de referencia
# =============================================================================
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(frames_rgb[0])
ax.set_title("Frame de referencia — Líneas de calibración y factor de escala interpolado", fontsize=13)

# --- Línea azul: referencia en y=345 (cercana) ---
r345 = ref[0]
ax.plot([r345["x0"], r345["x1"]], [r345["y"], r345["y"]], '-', color='cyan', linewidth=2.5)
ax.plot(r345["x0"], r345["y"], '+', color='cyan', markersize=14, markeredgewidth=2)
ax.plot(r345["x1"], r345["y"], '+', color='cyan', markersize=14, markeredgewidth=2)
ax.annotate(
    f'{r345["metros"]:.3f} m / {r345["px"]} px = {r345["escala"]:.6f} m/px',
    xy=((r345["x0"] + r345["x1"]) / 2, r345["y"]),
    xytext=(10, 15), textcoords="offset points",
    color='white', fontsize=9, fontweight='bold',
    bbox=dict(boxstyle='round,pad=0.3', fc='tab:blue', alpha=0.8),
    ha='center',
)

# --- Línea verde: referencia en y=260 (lejana) ---
r260 = ref[1]
ax.plot([r260["x0"], r260["x1"]], [r260["y"], r260["y"]], '-', color='lime', linewidth=2.5)
ax.plot(r260["x0"], r260["y"], '+', color='lime', markersize=14, markeredgewidth=2)
ax.plot(r260["x1"], r260["y"], '+', color='lime', markersize=14, markeredgewidth=2)
ax.annotate(
    f'{r260["metros"]:.4f} m / {r260["px"]} px = {r260["escala"]:.6f} m/px',
    xy=((r260["x0"] + r260["x1"]) / 2, r260["y"]),
    xytext=(10, -20), textcoords="offset points",
    color='white', fontsize=9, fontweight='bold',
    bbox=dict(boxstyle='round,pad=0.3', fc='green', alpha=0.8),
    ha='center',
)

# --- Línea roja: interpolación en y=286 ---
# Interpolar también x0 y x1 para dibujar una línea coherente a esa altura
x0_286 = int(r260["x0"] + (r345["x0"] - r260["x0"]) * (y_objetivo - y1) / (y2 - y1))
x1_286 = int(r260["x1"] + (r345["x1"] - r260["x1"]) * (y_objetivo - y1) / (y2 - y1))

ax.plot([x0_286, x1_286], [y_objetivo, y_objetivo], '--', color='red', linewidth=2.5)
ax.plot(x0_286, y_objetivo, '+', color='red', markersize=14, markeredgewidth=2)
ax.plot(x1_286, y_objetivo, '+', color='red', markersize=14, markeredgewidth=2)
ax.annotate(
    f'y={y_objetivo}: ≈ {escala_286:.6f} m/px (interpolado)',
    xy=((x0_286 + x1_286) / 2, y_objetivo),
    xytext=(10, -20), textcoords="offset points",
    color='white', fontsize=9, fontweight='bold',
    bbox=dict(boxstyle='round,pad=0.3', fc='tab:red', alpha=0.8),
    ha='center',
)

ax.axis("off")
plt.tight_layout()
plt.show()

## Cálculo de velocidad del objeto

Se filtran los frames cuyo bounding box **no toca los bordes laterales** de la imagen
(es decir, `x > 0` y `x + w < ancho`), asegurando que el objeto está completamente
visible y no cortado por los márgenes.

Luego se calcula la velocidad entre frames consecutivos válidos usando:
- Desplazamiento horizontal en píxeles → metros (con `escala_286`)
- Tiempo entre frames → `delta_frames / fps`
- Conversión final a km/h

In [ ]:
# =============================================================================
# Paso 1: Filtrar frames válidos (bbox no toca bordes laterales)
# =============================================================================

# Un frame es válido si el bbox detectado NO toca el borde izquierdo (x > 0)
# ni el borde derecho (x + w < NUEVO_ANCHO)
frames_validos = []
for i, bbox in enumerate(bboxes):
    if bbox is not None:
        x, y, w, h = bbox
        if x > 0 and (x + w) < NUEVO_ANCHO:
            frames_validos.append(i)

print(f"Frames con detección: {sum(1 for b in bboxes if b is not None)}")
print(f"Frames válidos (bbox sin tocar bordes laterales): {len(frames_validos)}")
print(f"  Índices: {frames_validos}")

# =============================================================================
# Paso 2: Calcular velocidad entre frames consecutivos válidos
# =============================================================================

velocidades = {}  # frame_idx -> velocidad en km/h

for i in range(1, len(frames_validos)):
    idx_prev = frames_validos[i - 1]
    idx_curr = frames_validos[i]

    bbox_prev = bboxes[idx_prev]
    bbox_curr = bboxes[idx_curr]

    if bbox_prev is None or bbox_curr is None:
        continue

    # Centro horizontal del bounding box
    cx_prev = bbox_prev[0] + bbox_prev[2] // 2
    cx_curr = bbox_curr[0] + bbox_curr[2] // 2

    # Desplazamiento en píxeles (eje X — movimiento horizontal)
    delta_px = abs(cx_curr - cx_prev)

    # Distancia real en metros
    delta_m = delta_px * escala_286

    # Tiempo entre los dos frames
    delta_frames = idx_curr - idx_prev
    delta_t = delta_frames / fps

    # Velocidad en m/s -> km/h
    velocidad_ms = delta_m / delta_t if delta_t > 0 else 0
    velocidad_kmh = velocidad_ms * 3.6

    velocidades[idx_curr] = velocidad_kmh

# Promedio de velocidad
velocidad_promedio = np.mean(list(velocidades.values())) if velocidades else 0.0

# =============================================================================
# Resumen
# =============================================================================
print(f"\n{'=' * 60}")
print("RESUMEN DE VELOCIDAD")
print(f"{'=' * 60}")
print(f"  Frames válidos analizados : {len(frames_validos)}")
if velocidades:
    vals = list(velocidades.values())
    print(f"  Velocidad mínima          : {min(vals):.2f} km/h")
    print(f"  Velocidad máxima          : {max(vals):.2f} km/h")
    print(f"  Velocidad promedio        : {velocidad_promedio:.2f} km/h")
else:
    print("  No se pudieron calcular velocidades.")

In [ ]:
plt.close('all')

# =============================================================================
# Gráfica de velocidad vs. frame
# =============================================================================
if velocidades:
    frames_vel = sorted(velocidades.keys())
    vals_vel = [velocidades[f] for f in frames_vel]

    fig, ax = plt.subplots(figsize=(12, 5))

    # Línea de velocidad
    ax.plot(frames_vel, vals_vel, 'o-', color='tab:red', markersize=5, label='Velocidad instantánea')

    # Línea promedio
    ax.axhline(y=velocidad_promedio, color='tab:green', linestyle='--', linewidth=1.5,
               label=f'Promedio: {velocidad_promedio:.2f} km/h')

    # Marcar frames válidos en el eje X
    ax.set_xlabel("Frame", fontsize=12)
    ax.set_ylabel("Velocidad (km/h)", fontsize=12)
    ax.set_title("Velocidad del objeto detectado entre frames válidos", fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No hay velocidades calculadas para graficar.")

## GIF animado de la detección con velocidad

Se genera un GIF que muestra 3 paneles simultáneos para cada frame:
- **Original**: frame RGB
- **Máscara**: resultado de diferencia + umbral + morfología
- **Detección**: frame con bounding box del objeto detectado

Cada frame incluye un **pie de foto** con la velocidad instantánea (si fue calculada) y el promedio general.

In [ ]:
plt.close('all')

from PIL import Image as PILImage
import io

frames_gif = []

for i in range(len(frames_rgb)):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Panel 1: Frame original
    axes[0].imshow(frames_rgb[i])
    axes[0].set_title(f'Frame {i} - Original')
    axes[0].axis('off')

    # Panel 2: Máscara binaria
    axes[1].imshow(mascaras[i], cmap='gray')
    axes[1].set_title('Máscara')
    axes[1].axis('off')

    # Panel 3: Detección con bounding box
    axes[2].imshow(frames_rgb[i])
    bbox = bboxes[i]
    if bbox is not None:
        x, y, w, h = bbox
        rect = Rectangle((x, y), w, h, linewidth=2, edgecolor='lime', facecolor='none')
        axes[2].add_patch(rect)
        cx, cy = x + w // 2, y + h // 2
        axes[2].plot(cx, cy, '+', color='red', markersize=10, markeredgewidth=2)
        axes[2].set_title(f'Detección — centro: ({cx}, {cy})')
    else:
        axes[2].set_title('Sin detección')
    axes[2].axis('off')

    # Pie de foto con velocidad
    if i in velocidades:
        vel_texto = f"Velocidad: {velocidades[i]:.2f} km/h"
    else:
        vel_texto = "Velocidad: —"

    fig.text(
        0.5, 0.02,
        f"{vel_texto}  |  Promedio: {velocidad_promedio:.2f} km/h",
        ha='center', fontsize=12, color='white',
        bbox=dict(boxstyle='round', facecolor='black', alpha=0.7)
    )

    plt.tight_layout(rect=[0, 0.06, 1, 1])  # Dejar espacio para el pie de foto

    # Convertir figura a imagen PIL
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=80)
    buf.seek(0)
    frames_gif.append(PILImage.open(buf).copy())
    plt.close(fig)

# Guardar GIF con velocidad
gif_path = 'velocidad.gif'
frames_gif[0].save(
    gif_path,
    save_all=True,
    append_images=frames_gif[1:],
    duration=100,  # ms por frame (~10 fps)
    loop=0
)

print(f"GIF guardado en: {gif_path} ({len(frames_gif)} frames)")

# Mostrar el GIF en el notebook
from IPython.display import Image as IPImage
IPImage(gif_path)